# TopN Rolling Backtest Notebook

This notebook evaluates the **winner-bucket strategy** using a TopN rolling portfolio:

- Signal: `winner_prob` or rank score per (date, ticker)
- Selection: Top5 / Top10 / Top20
- Holding: fixed 10 trading days, rolling sleeves
- Primary metric: benchmark-relative excess alpha
- Risk constraints: Sharpe / Max Drawdown / Turnover
- Secondary metric: TopN-BottomN spread


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.research.topn_backtest import run_topn_rolling_backtest

## 1. Load signal and realised returns

In [ ]:
# Replace these synthetic placeholders with your actual outputs
dates = pd.bdate_range('2024-01-02', periods=200)
tickers = [f'S{i:03d}' for i in range(60)]
rng = np.random.default_rng(42)

signal_rows = []
for d in dates:
    scores = rng.normal(size=len(tickers))
    for t, s in zip(tickers, scores, strict=False):
        signal_rows.append({'date': d, 'ticker': t, 'winner_prob': float(s)})
signal_df = pd.DataFrame(signal_rows)

realized_returns = pd.DataFrame(
    rng.normal(0.0005, 0.02, size=(len(dates), len(tickers))),
    index=dates, columns=tickers
)
bench_returns = pd.Series(rng.normal(0.0002, 0.01, size=len(dates)), index=dates, name='bench')

signal_df.head()

## 2. Run TopN backtests

In [ ]:
results = {}
portfolios = {}
for k in [5, 10, 20]:
    portfolio_df, summary = run_topn_rolling_backtest(
        signal_df=signal_df,
        realized_returns=realized_returns,
        bench_returns=bench_returns,
        score_col='winner_prob',
        top_n=k,
        holding_days=10,
    )
    portfolios[k] = portfolio_df
    results[k] = summary.__dict__

summary_df = pd.DataFrame(results).T
summary_df

## 3. Plot cumulative excess alpha

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for k, pf in portfolios.items():
    (1 + pf['excess_alpha'].fillna(0)).cumprod().plot(ax=ax, label=f'Top{k}')
ax.set_title('Cumulative Excess Alpha by TopN Basket')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Plot cumulative Top-Bottom spread

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for k, pf in portfolios.items():
    (1 + pf['top_bottom_spread'].fillna(0)).cumprod().plot(ax=ax, label=f'Top{k}-Bottom{k}')
ax.set_title('Cumulative Top-Bottom Spread')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Inspect turnover

In [ ]:
turnover_df = pd.DataFrame({f'Top{k}': pf['turnover'] for k, pf in portfolios.items()})
turnover_df.describe().T

## Notes

- This notebook uses a **rolling sleeve** construction, not a single-period static basket.
- `winner_prob` can come from `LGBMClassifier` or any rank-compatible scoring model.
- Primary go/no-go metric remains **TopN excess alpha**, with spread as a secondary diagnostic.
